In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
#dir_in = '/home2/DATA_AO/CGPS_GMIMS'
#dir_in = '/home/aordog/DATA/CGPS_GMIMS/'
#dir_in = '/srv/aordog/cgps_gmims_data/'
dir_in = '/srv/data/cgps-gmims/conv_regrid/'

## Read in RM files:

In [ ]:
hdu_CG_RM = fits.open(dir_in+'RM_CG_conv4_regrd.fits')
CG_RM = hdu_CG_RM[0].data
print(CG_RM.shape)

hdu_G_RM = fits.open(dir_in+'RM_G_regrd.fits')
G_RM = hdu_G_RM[0].data
print(G_RM.shape)

hdu_C_RM = fits.open(dir_in+'RM_C_conv4_regrd.fits')
C_RM = hdu_C_RM[0].data
print(C_RM.shape)

rvalue_CG = hdu_CG_RM[2].data
rvalue_G  = hdu_G_RM[2].data
rvalue_C  = hdu_C_RM[2].data

stderr_CG = hdu_CG_RM[4].data
stderr_G  = hdu_G_RM[4].data
stderr_C  = hdu_C_RM[4].data

PAint_CG  = hdu_CG_RM[1].data
PAint_G   = hdu_G_RM[1].data
PAint_C   = hdu_C_RM[1].data

## Read in PI files:

In [ ]:
hdu_CG_PI = fits.open(dir_in+'PI_CG_conv4_regrd_PI_of_mean.fits')
CG_PI = hdu_CG_PI[0].data
print(CG_PI.shape)

hdu_G_PI = fits.open(dir_in+'PI_G_regrd_PI_of_mean.fits')
G_PI = hdu_G_PI[0].data
print(G_PI.shape)

hdu_C_PI = fits.open(dir_in+'PI_C_conv4_regrd_PI_of_mean.fits')
C_PI = hdu_C_PI[0].data
print(C_PI.shape)  

## Read in PA files:

In [ ]:
PA_CG_list = []
PA_G_list = []
PA_C_list = []

band = ['A','B','C','D']
bandlc = ['a','b','c','d']
for i in range(0,4):
    
    print('band '+band[i])
    
    # CGPS + GMIMS (CG)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_CG = fits.open(directory+'PA_'+band[i]+'_CG_conv4_regrd.fits')
    PA_CG_list.append(hdu_CG)
    
    # Grab a header (all the same for now)
    hdr = hdu_CG[0].header
    
    # CGPS only (C)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_C = fits.open(directory+'PA_'+band[i]+'_C_conv4_regrd.fits')
    PA_C_list.append(hdu_C)
    
    # GMIMS only (G)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_G = fits.open(directory+'PA_'+band[i]+'_G_regrd.fits')
    PA_G_list.append(hdu_G)
    
gc.collect()


## Individual band PI:

In [ ]:
band_PI = False

if band_PI:

    CG_PI_list = []
    G_PI_list = []
    C_PI_list = []

    band = ['A','B','C','D']
    bandlc = ['a','b','c','d']
    for i in range(0,4):

        hdu_CG_PI = fits.open(dir_in+'PI'+band[i]+'_CG.fits')
        hdu_G_PI = fits.open(dir_in+'PI'+band[i]+'_G.fits')
        hdu_C_PI = fits.open(dir_in+'PI'+band[i]+'_C.fits')
        CG_PI_list.append(hdu_CG_PI[0].data)
        G_PI_list.append(hdu_G_PI[0].data)
        C_PI_list.append(hdu_C_PI[0].data)

In [ ]:
#G_PI = np.nanmean(np.array(G_PI_list),axis=0)
#C_PI = np.nanmean(np.array(C_PI_list),axis=0)
#CG_PI = np.nanmean(np.array(CG_PI_list),axis=0)

In [ ]:
#del CG_PI_list, G_PI_list,C_PI_list
#gc.collect()

## Set PI threshold:

In [ ]:
PI_lim = 0.05

wGbad = np.where(G_PI < PI_lim)
wCGbad = np.where(CG_PI < PI_lim)

#CG_RM_good = CG_RM.copy()
#CG_RM_good[wCGbad] = np.nan

#C_RM_good = C_RM.copy()
#C_RM_good[wCGbad] = np.nan

G_RM_good = G_RM.copy()
G_RM_good[wGbad] = np.nan

del wGbad,wCGbad
gc.collect()

## Plot all PI and RM

In [ ]:
fs = 18
fig = plt.figure(figsize=(20, 16))

PIlim = 0.4
RMlim = 400

c = SkyCoord([100,95], [-1,3], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(331,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(332,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(333,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(334,projection=wcs[j1:j2,i1:i2])
#plt.imshow(G_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(335,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(336,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(337,projection=wcs[j1:j2,i1:i2])
#plt.imshow(G_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(338,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(339,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

## Full CGPS plots for poster

In [ ]:
fs = 18
fig = plt.figure(figsize=(80, 20))
fig.subplots_adjust(left=0.05, right=0.99, top=0.99, bottom=0.05,hspace=0.001)

PIlim1 = 0.5
RMlim = 300
PI_lim = 0.1
fraction = 0.08
pad = 0.005
asp = 10

c = SkyCoord([100,52], [-3,5], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower',cmap='gray')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='red', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp,label='PI (K)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(412,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(C_RM[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(413,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(G_RM_good[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(414,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_RM[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('Galactic Longitude',fontsize=fs)
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.tight_layout()
plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/mask_test.png')


probably best option is: combined PI, SA-RM, ST-RM, ST+SA-RM
maybe 1D RM vs long for both ST and SA+ST
(and all of these with convolved versions, not raw)

## Make sample PA verus lambda^2 plots

In [ ]:
def read_in_QU_files():
    CG_Q_list = []
    CG_U_list = []
    G_Q_list = []
    G_U_list = []
    C_Q_list = []
    C_U_list = []

    band = ['A','B','C','D']
    bandlc = ['a','b','c','d']
    for i in range(0,4):
        
        print('band '+band[i])
        
        # CGPS + GMIMS (CG)
        #directory = '/srv/aordog/cgps_gmims_data/'
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_CG_Q = fits.open(directory+'Q'+band[i]+'_CG_conv4_regrd.fits')
        hdu_CG_U = fits.open(directory+'U'+band[i]+'_CG_conv4_regrd.fits')
        CG_Q_list.append(hdu_CG_Q[0].data)
        CG_U_list.append(hdu_CG_U[0].data)
        
        # Grab a header (all the same for now)
        hdr = hdu_CG_Q[0].header
        
        # CGPS only (C)
        #directory = '/srv/aordog/cgps_gmims_data/'
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_C_Q = fits.open(directory+'Q'+band[i]+'_C_conv4_regrd.fits')
        hdu_C_U = fits.open(directory+'U'+band[i]+'_C_conv4_regrd.fits')
        C_Q_list.append(hdu_C_Q[0].data)
        C_U_list.append(hdu_C_U[0].data)
        
        # GMIMS only (G)
        #directory = '/srv/aordog/cgps_gmims_data/'
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_G_Q = fits.open(directory+'Q'+band[i]+'_G_regrd.fits')
        hdu_G_U = fits.open(directory+'U'+band[i]+'_G_regrd.fits')
        G_Q_list.append(hdu_G_Q[0].data)
        G_U_list.append(hdu_G_U[0].data)
        
    gc.collect()

    return CG_Q_list, CG_U_list, G_Q_list, G_U_list, C_Q_list, C_U_list, hdr

def make_PA_arrays(CG_Q_list,CG_U_list,G_Q_list,G_U_list,C_Q_list,C_U_list,
                   l_idx0,b_idx0):
    Q0_C = np.empty([4])
    U0_C = np.empty([4])
    Q0_G = np.empty([4])
    U0_G = np.empty([4])
    Q0_CG = np.empty([4])
    U0_CG = np.empty([4])

    for i in range(0,4):
        Q0_C[i] = C_Q_list[i][b_idx0,l_idx0]
        U0_C[i] = C_U_list[i][b_idx0,l_idx0]
        Q0_G[i] = G_Q_list[i][b_idx0,l_idx0]
        U0_G[i] = G_U_list[i][b_idx0,l_idx0]
        Q0_CG[i] = CG_Q_list[i][b_idx0,l_idx0]
        U0_CG[i] = CG_U_list[i][b_idx0,l_idx0]
    
    PA_C = 0.5*np.arctan2(U0_C,Q0_C)
    PA_G = 0.5*np.arctan2(U0_G,Q0_G)
    PA_CG = 0.5*np.arctan2(U0_CG,Q0_CG)

    return PA_C, PA_G, PA_CG

def unwrap_angles(PA_raw):

    zz = np.exp(2*PA_raw*1j)
    dtau = np.empty([3])

    for i in range(0,3):
        dtau[i] = np.arctan2((zz[i+1]*np.conj(zz[i])).imag,
                             (zz[i+1]*np.conj(zz[i])).real)/2.
    PA_fix = PA_raw.copy()
    for i in range(1,4):
        PA_fix[i] = PA_fix[i-1] + dtau[i-1]

    return PA_fix

In [ ]:
plot_lines = True

if plot_lines:

    freq = np.array([1406.9,1413.8,1427.4,1434.3])
    lbd2 = ((3e8)/(freq*1e6))**2

    CG_Q_list,CG_U_list,G_Q_list,G_U_list,C_Q_list,C_U_list,hdr = read_in_QU_files()

    l_idx = np.linspace(0,hdr['NAXIS1']-1,hdr['NAXIS1'])
    b_idx = np.linspace(0,hdr['NAXIS2']-1,hdr['NAXIS2'])
    l = hdr['CRVAL1']+(l_idx - hdr['CRPIX1']+1)*hdr['CDELT1']
    b = hdr['CRVAL2']+(b_idx - hdr['CRPIX2']+1)*hdr['CDELT2']

    dlb = l[0]-l[1]

    print(l)
    print(b)

In [ ]:
from scipy.stats import pearsonr

In [ ]:
def make_4panel_linfit(l_list, b_list):

    fs = 14
    s = 50
    lbd2_ext = np.linspace(0.0435,0.0456,100)

    fig, axs = plt.subplots(2,2,figsize=(12,12))
    plt.subplots_adjust(left=0.1, bottom=0.05, right=0.99, top=0.95, wspace=0.2, hspace=0.2)

    for i in range(0,2):
        for j in range(0,2):
            
            k = 2*i + j
            print(k)

            l_idx0 = np.where(abs(l-l_list[k])<dlb/2.)[0][0]
            b_idx0 = np.where(abs(b-b_list[k])<dlb/2.)[0][0]

            print(l[l_idx0])
            print(b[b_idx0])

            #PA_C,PA_G,PA_CG = make_PA_arrays(CG_Q_list,CG_U_list,G_Q_list,G_U_list,
            #                                 C_Q_list,C_U_list,l_idx0,b_idx0)
    
            #PA_C_fix = unwrap_angles(PA_C)
            #PA_G_fix = unwrap_angles(PA_G)
            #PA_CG_fix = unwrap_angles(PA_CG)

            PA_C_fix = []
            PA_G_fix = []
            PA_CG_fix = []
            PA_C_err = []
            PA_G_err = []
            PA_CG_err = []

            for k in range(0,4):

                PA_C_fix.append(PA_C_list[k][0].data[b_idx0,l_idx0])
                PA_G_fix.append(PA_G_list[k][0].data[b_idx0,l_idx0])
                PA_CG_fix.append(PA_CG_list[k][0].data[b_idx0,l_idx0])

                PA_C_err.append(PA_C_list[k][1].data[b_idx0,l_idx0])
                PA_G_err.append(PA_G_list[k][1].data[b_idx0,l_idx0])
                PA_CG_err.append(PA_CG_list[k][1].data[b_idx0,l_idx0])

            PAint_pt_G = PAint_G[b_idx0,l_idx0]
            PAint_pt_C = PAint_C[b_idx0,l_idx0]
            PAint_pt_CG = PAint_CG[b_idx0,l_idx0]
            RM_pt_G = G_RM[b_idx0,l_idx0]
            RM_pt_C = C_RM[b_idx0,l_idx0]
            RM_pt_CG = CG_RM[b_idx0,l_idx0]
            err_pt_CG = stderr_CG[b_idx0,l_idx0]
            err_pt_G = stderr_G[b_idx0,l_idx0]
            err_pt_C = stderr_C[b_idx0,l_idx0]

            axs[i,j].errorbar(lbd2,np.array(PA_C_fix)*180/np.pi,yerr=np.array(PA_C_err)*180/np.pi,color='blue',fmt="o",capsize=5)
            axs[i,j].errorbar(lbd2,np.array(PA_G_fix)*180/np.pi,yerr=np.array(PA_G_err)*180/np.pi,color='orange',fmt="o",capsize=5)
            axs[i,j].errorbar(lbd2,np.array(PA_CG_fix)*180/np.pi,yerr=np.array(PA_CG_err)*180/np.pi,color='red',fmt="o",capsize=5)

            axs[i,j].plot(lbd2_ext,PAint_pt_C*180/np.pi+RM_pt_C*lbd2_ext*180/np.pi,color='blue',
                          label='RM$_{AS}$ = '+str(np.round(RM_pt_C,1))+'$\pm$'+str(np.round(err_pt_C,1))+' rad m$^{-2}$')
            
            axs[i,j].plot(lbd2_ext,PAint_pt_G*180/np.pi+RM_pt_G*lbd2_ext*180/np.pi,color='orange',
                          label='RM$_{SA}$ = '+str(np.round(RM_pt_G,1))+'$\pm$'+str(np.round(err_pt_G,1))+' rad m$^{-2}$')
            
            axs[i,j].plot(lbd2_ext,PAint_pt_CG*180/np.pi+RM_pt_CG*lbd2_ext*180/np.pi,color='red',
                          label='RM$_{SA+AS}$ = '+str(np.round(RM_pt_CG,1))+'$\pm$'+str(np.round(err_pt_CG,1))+' rad m$^{-2}$')
            
            #axs[i,j].set_ylim(np.min(PA_G_fix*180/np.pi),np.max(PA_G_fix*180/np.pi))
            axs[i,j].set_xlim(0.0434,0.0456)
            axs[i,j].set_xticks([0.0435,0.044,0.0445,0.045,0.0455])
            axs[i,j].set_yticks([-180,-135,-90,-45,0,45,90,135,180])
            axs[1,j].set_xlabel(r'$\lambda^2$ (m$^2$)',fontsize=fs)
            axs[i,0].set_ylabel('PA (deg.)',fontsize=fs)
            axs[i,j].grid()
            axs[i,j].tick_params(axis='both', labelsize=fs)
            axs[i,j].legend(fontsize=fs)
            axs[i,j].set_title('$\ell$ = '+str(np.round(l[l_idx0],3))+r'$^{\circ}$, $b$ = '+str(np.round(b[b_idx0],3))+r'$^{\circ}$',
                              fontsize=fs+2)

    axs[0,0].set_ylim(-45,135)
    axs[0,1].set_ylim(-180,0)
    axs[1,0].set_ylim(-180,0)
    axs[1,1].set_ylim(-180,0)

    return

In [ ]:
l_list = [125.79, 188.84, 93.56, 134.61]
b_list = [0.97,   2.54,  -0.32,  -1.51]

make_4panel_linfit(l_list,b_list)

plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/sample_linear_fits_errorbars.pdf')
#plt.tight_layout()

In [ ]:
###############
l0 = 136.9
b0 = 0.58
###############

if plot_lines:

    l_idx0 = np.where(abs(l-l0)<dlb/2.)[0][0]
    b_idx0 = np.where(abs(b-b0)<dlb/2.)[0][0]

    print(l[l_idx0])
    print(b[b_idx0])

    PA_C,PA_G,PA_CG = make_PA_arrays(CG_Q_list,CG_U_list,G_Q_list,G_U_list,
                                     C_Q_list,C_U_list,l_idx0,b_idx0)
    
    PA_C_fix = unwrap_angles(PA_C)
    PA_G_fix = unwrap_angles(PA_G)
    PA_CG_fix = unwrap_angles(PA_CG)
    
    #plt.scatter(lbd2,PA_C*180/np.pi,s=80)
    #plt.scatter(lbd2,PA_C_fix*180/np.pi,color='blue',s=20)
    plt.scatter(lbd2,PA_G*180/np.pi,s=80,color='C1')
    plt.scatter(lbd2,PA_G_fix*180/np.pi,color='red',s=20)
    #plt.scatter(lbd2,PA_CG*180/np.pi,s=80)
    #plt.scatter(lbd2,PA_CG_fix*180/np.pi,color='darkgreen',s=20)
    plt.ylim(np.min(PA_G_fix*180/np.pi),np.max(PA_G_fix*180/np.pi))
    plt.xlim(0.0435,0.046)
    plt.grid()
    
    print('------')
    print(np.round(PA_G*180/np.pi,1))
    print(np.round(PA_G[1:4]*180/np.pi - PA_G[0:3]*180/np.pi,1))
    print('')
    print(np.round(PA_G_fix*180/np.pi,1))
    print(np.round(PA_G_fix[1:4]*180/np.pi - PA_G_fix[0:3]*180/np.pi,1))
    
    linfit = np.polyfit(lbd2,PA_G_fix,1)[0]
    print('')
    print(linfit)
    print(pearsonr(lbd2,PA_G_fix)[0])


In [ ]:
b_arr = np.linspace(-0.1,0.9,3)
l0 = 72.0

for b0 in b_arr:

    l_idx0 = np.where(abs(l-l0)<dlb/2.)[0][0]
    b_idx0 = np.where(abs(b-b0)<dlb/2.)[0][0]

    #print(l[l_idx0])
    #print(b[b_idx0])

    PA_C,PA_G,PA_CG = make_PA_arrays(CG_Q_list,CG_U_list,G_Q_list,G_U_list,
                                     C_Q_list,C_U_list,l_idx0,b_idx0)
    
    PA_C_fix = unwrap_angles(PA_C)
    PA_G_fix = unwrap_angles(PA_G)
    PA_CG_fix = unwrap_angles(PA_CG)
    
    #plt.scatter(lbd2,PA_C*180/np.pi,s=80)
    #plt.scatter(lbd2,PA_C_fix*180/np.pi,color='blue',s=20)
    plt.scatter(lbd2,PA_G*180/np.pi,s=80,color='C0',zorder=0)
    plt.scatter(lbd2,PA_G_fix*180/np.pi,color='blue',s=20,zorder=500)
    #plt.scatter(lbd2,PA_CG*180/np.pi,s=80)
    #plt.scatter(lbd2,PA_CG_fix*180/np.pi,color='darkgreen',s=20)
plt.ylim(-250,250)
plt.xlim(0.0435,0.046)
plt.grid()

In [ ]:
50*(np.pi/180)/(lbd2[0]-lbd2[3])

In [ ]:
fs = 18
fig = plt.figure(figsize=(80, 15))

PIlim1 = 0.5
RMlim = 300
PI_lim = 0.1
fraction = 0.08
pad = 0.005
asp = 10

c = SkyCoord([192,52], [-3,5], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower',cmap='gray')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='red', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
#plt.yticks(fontsize=fs)
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp,label='PI (K)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.tight_layout()
#plt.savefig('../CGPS_GMIMS_plots/poster_test.png')

In [ ]:
print(wcs)

In [ ]:
print(ax.xaxis)

In [ ]:
data = np.random.rand(10, 10)

# Create an image plot of the data
fig = plt.subplots()
plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower')

# Add a colorbar to the plot
#divider = make_axes_locatable(plt.gca())
#cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im)
#cbar.ax.tick_params(labelsize=10)

In [ ]:
print(plt.gca()[0])

In [ ]:
fs = 24
fig = plt.figure(figsize=(20, 8))

PIlim = 0.4
RMlim = 400

c = SkyCoord([90,55], [-5,8], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

fraction = 0.018
pad = 0.005
asp = 20

plt.subplot(111,projection=wcs[j1:j2,i1:i2])
#im = plt.imshow(C_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
im = plt.imshow(CG_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude',fontsize=fs)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.tick_params(axis='both', which='major', labelsize=fs)

cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.savefig('/home/aordog/Python/cgps-gmims/CGPS_GMIMS_example_map.jpg')